# 00. Data exploration, monthly audit and merged-data export

Run this notebook first, then `01_crime_original.ipynb` and `02_crime_mps.ipynb`. Both primary analysis notebooks read `data/crime_housing_merged.csv.gz`. Their model numbering is consistent and supersedes earlier exploratory notebook labels.

Inputs may be directly in `data/` or, locally, `project_data/`. The original sources are never overwritten. Original crime is constant within LSOA-year but has suspicious endpoint counts and all-zero violence in 2014-2015. We document these issues without asserting an unverified partial-year explanation. External offence groups may differ from the supplied definitions.


In [1]:
from pathlib import Path
try:
    from google.colab import drive
except ImportError:
    ROOT = Path.cwd()
else:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/HE3027 Project')
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

def find_input(filename):
    for folder in [DATA_DIR, ROOT / 'project_data', ROOT]:
        path = folder / filename
        if path.is_file():
            return path
    raise FileNotFoundError('Cannot find ' + filename)
DATA = find_input('land_registry_crime.dta')
MONTHLY_FILE = find_input('MPS LSOA Level Crime (Historical).csv')
OUT = DATA_DIR / 'audit_outputs'
OUT.mkdir(exist_ok=True)
print('Original:', DATA, '\nMonthly:', MONTHLY_FILE)


Original: /home/rachmiel/he3027/project_data/land_registry_crime.dta 
Monthly: /home/rachmiel/he3027/project_data/MPS LSOA Level Crime (Historical).csv


In [2]:
import hashlib
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
CRIMES = ['allcrime', 'violent_crime', 'propertycrime']
pd.set_option('display.max_columns', 12)
pd.set_option('display.precision', 4)

def table(data, filename):
    """Save the complete table and display a compact preview."""
    data.to_csv(OUT / (filename + '.csv'))
    display(data.head(20) if len(data) > 20 else data)
    if len(data) > 20:
        print(f'Showing 20 of {len(data)} rows; the complete table is saved as {filename}.csv.')

VARIABLE_LABELS = {
    'detached_dum': 'Detached (relative to flat)',
    'semi_d_dum': 'Semi-detached (relative to flat)',
    'terrace_dum': 'Terraced (relative to flat)',
    'freehold': 'Freehold',
    'newbuild': 'New build',
    'tube_distnear_km': 'Distance to tube station (km)',
    'bus_distnear_km': 'Distance to bus stop (km)',
    'thamesriv_dist_km': 'Distance to Thames (km)',
    'Nearest_Park_dist_km': 'Distance to park (km)',
    'dist_to_cbd': 'Distance to CBD (km)',
    'Bus_stopscount_200m': 'Bus stops within 200m',
    'heritage_count_200m': 'Heritage buildings within 200m',
    'percent_noedu': 'Residents without qualifications',
    'percent_minorities': 'Minority residents',
    'lone_hh': 'Lone-parent households',
    'lnpay': 'Log borough annual pay',
    'jobdensity': 'Borough job density',
    'hoursworked': 'Borough weekly hours worked',
    'unemployment': 'Borough unemployment',
}

def coefficient_inference(coefficients, results):
    """Compute inference for every coefficient from its saved clustered SE."""
    keys = [name for name in ['panel', 'model', 'crime', 'measure', 'timing']
            if name in coefficients and name in results]
    metadata = results[keys + ['clusters']]
    enriched = coefficients.drop(columns=['clusters', 't', 'p', 'ci_low', 'ci_high'], errors='ignore')
    enriched = enriched.merge(metadata, on=keys, how='left', validate='many_to_one')
    if enriched.clusters.isna().any():
        raise ValueError('Coefficient rows could not be matched to regression metadata.')
    degrees_freedom = enriched.clusters - 1
    enriched['t'] = enriched.beta / enriched.se
    enriched['p'] = 2 * stats.t.sf(np.abs(enriched.t), degrees_freedom)
    critical = stats.t.ppf(0.975, degrees_freedom)
    enriched['ci_low'] = enriched.beta - critical * enriched.se
    enriched['ci_high'] = enriched.beta + critical * enriched.se
    return enriched

def show_models(results, filename, coefficient_filename):
    """Show one expandable table per model with crime AND control coefficients."""
    from html import escape
    from IPython.display import HTML

    results.to_csv(OUT / (filename + '.csv'), index=False)
    coefficients = coefficient_inference(pd.read_csv(OUT / coefficient_filename), results)
    coefficients.to_csv(OUT / coefficient_filename, index=False)
    cards = []

    def number(value):
        if pd.isna(value):
            return '—'
        if value != 0 and abs(value) < 0.0001:
            return f'{value:.2e}'
        return f'{value:.4f}'

    for _, regression in results.iterrows():
        selected = coefficients.model.eq(regression['model'])
        for key in ['panel', 'crime', 'measure', 'timing']:
            if key in coefficients and key in results:
                selected &= coefficients[key].eq(regression[key])
        rows = coefficients.loc[selected].copy()
        title = regression['model'] + ' · ' + str(regression.get('crime', regression.get('measure', '')))
        if 'timing' in results:
            title += ' · ' + regression['timing']
        rendered = []
        for _, row in rows.iterrows():
            label = VARIABLE_LABELS.get(row.variable, row.variable)
            if row.variable.startswith('log1p_'):
                label = 'Crime: log(1 + ' + row.variable[6:].replace('_', ' ') + ')'
            rendered.append({
                'Variable': label,
                'Coefficient': number(row.beta),
                'Clustered SE': number(row.se),
                't statistic': f'{row.t:.2f}',
                '95% CI': number(row.ci_low) + ' to ' + number(row.ci_high),
                'p-value': '<0.001' if row.p < 0.001 else f'{row.p:.3f}',
            })
        effects = regression.get('fixed_effects', '')
        effects = 'None' if pd.isna(effects) or not effects else str(effects)
        r2_label = 'Within R²' if regression.r2_type == 'within' else 'R²'
        subtitle = (f'N = {int(regression.N):,} | LSOA clusters = {int(regression.clusters):,}'
                    f' | {r2_label} = {regression.r2:.4f}')
        body = pd.DataFrame(rendered).to_html(index=False, border=0, escape=True)
        omitted = regression.get('absorbed_or_collinear', '')
        note = '' if pd.isna(omitted) or not omitted else '<p>Absorbed/collinear: ' + escape(str(omitted)) + '</p>'
        cards.append('<details class="reg-card"><summary>' + escape(title) + '</summary>'
                     '<p>' + escape(subtitle) + '</p><p>Fixed effects: ' + escape(effects) + '</p>'
                     + body + note + '</details>')
    style = '<style>.reg-card{margin:10px 0;padding:12px;border:1px solid #bbb;border-radius:6px;overflow-x:auto}.reg-card summary{cursor:pointer;font-weight:600}.reg-card table{border-collapse:collapse;width:100%;max-width:950px}.reg-card th,.reg-card td{padding:7px 10px;border-bottom:1px solid #ddd;text-align:right;white-space:nowrap}.reg-card th:first-child,.reg-card td:first-child{text-align:left}.reg-card p{font-size:13px}</style>'
    display(HTML(style + '<p>Expand a regression to view all estimated coefficients. Standard errors are clustered by LSOA.</p>' + ''.join(cards)))
    return coefficients


In [3]:
df = pd.read_stata(DATA, convert_categoricals=False)
df['transaction_id'] = np.arange(len(df))
with pd.io.stata.StataReader(DATA) as reader:
    labels = reader.variable_labels()
print(f'{len(df):,} transactions and {len(labels)} original variables')
print('Dates:', df.transactiondates.min(), 'to', df.transactiondates.max())
pd.Series(labels, name='stata_label').to_csv(OUT / 'variable_labels.csv')
pd.Series({'data_sha256': hashlib.sha256(DATA.read_bytes()).hexdigest(), 'pandas': pd.__version__, 'numpy': np.__version__}).to_csv(OUT / 'provenance.csv')


549,813 transactions and 50 original variables
Dates: 2010-01-01 00:00:00 to 2015-07-30 00:00:00


## 1. Transaction structure, missingness and supplied crime

Counts below treat each LSOA-year once when describing crime, rather than weighting areas by the number of house sales. Zeros and missing values remain distinct. All transactions are retained in the export; analysis sample flags apply later.


In [4]:
crime_constancy = df.groupby(['lsoacode', 'year'])[CRIMES].nunique()
assert not crime_constancy.gt(1).any().any(), 'Crime varies within LSOA-year: inspect before modelling.'
area_year = df.groupby(['lsoacode', 'year'], as_index=False)[CRIMES].first()
table(area_year[CRIMES].describe().T, 'lsoa_year_crime_descriptives')
table(area_year.groupby('year')[CRIMES].agg(['mean', 'median', lambda s: s.eq(0).mean()]), 'crime_by_year_with_zero_shares')
print('SOURCE VALIDATION WARNING: crime means are much lower in 2010 and 2015;')
print('violent crime is zero throughout the observed 2014 and 2015 LSOA-years.')
print('Do not infer full-year comparability or interpret these zeros as validated absence of violence.')


,count,mean,std,min,25%,50%,75%,max
allcrime,28858.0,154.3839,258.5377,1.0,20.0,108.0,199.0,9225.0
violent_crime,28858.0,11.6884,23.4100,0.0,0.0,3.0,15.0,1098.0
propertycrime,28858.0,19.7013,19.2776,0.0,3.0,18.0,30.0,389.0


allcrime                   violent_crime                    \
          mean median <lambda_0>          mean median <lambda_0>   
year                                                               
2010   19.3462   14.0        0.0        2.4603    2.0     0.2412   
2011  255.0868  186.0        0.0       30.8135   23.0     0.0015   
2012  233.6485  172.0        0.0       28.1486   22.0     0.0010   
2013  205.8616  150.0        0.0        8.6017    7.0     0.0286   
2014  195.1985  142.0        0.0        0.0000    0.0     1.0000   
2015   15.1133   11.0        0.0        0.0000    0.0     1.0000   

     propertycrime                    
              mean median <lambda_0>  
year                                  
2010        1.6447    1.0     0.2870  
2011       24.5559   22.0     0.0008  
2012       32.1349   29.0     0.0002  
2013       29.6725   26.0     0.0008  
2014       27.5974   24.0     0.0002  
2015        2.3272    2.0     0.1701

SOURCE VALIDATION WARNING: crime means are much lower in 2010 and 2015;
violent crime is zero throughout the observed 2014 and 2015 LSOA-years.
Do not infer full-year comparability or interpret these zeros as validated absence of violence.


In [5]:
table(df[['price','allcrime','violent_crime','propertycrime','grossannualpay','jobdensity','hoursworked','unemployment']].describe().T, 'transaction_descriptives')
missing = df.isna().mean().sort_values(ascending=False)
table(missing.to_frame('missing_share'), 'missingness')
table(df.groupby('year').agg(sales=('price','size'), median_price=('price','median'), lsoas=('lsoacode','nunique')), 'sales_by_year')
table(df.groupby('propertyty').size().to_frame('sales'), 'property_types')
table(df.groupby('lsoacode')[CRIMES].agg(['mean','std']), 'within_lsoa_crime_variation')
fig, axes = plt.subplots(1, 3, figsize=(15,4))
df.groupby('year').price.median().plot(ax=axes[0], marker='o', title='Median transaction price')
axes[0].set_ylabel('GBP')
area_year.groupby('year')[CRIMES].mean().plot(ax=axes[1], marker='o', title='Supplied counts: mean LSOA-year')
missing.head(10).sort_values().plot.barh(ax=axes[2], title='Highest missing shares')
fig.tight_layout(); fig.savefig(OUT / 'original_data_audit.png', dpi=160); plt.show()


,count,mean,std,min,25%,50%,75%,max
price,549813.0,463444.2778,607860.5786,10000.00,235000.00,322450.00,490000.0000,5.4959e+07
allcrime,549813.0,182.6549,315.0519,1.00,43.00,123.00,216.0000,9.2250e+03
violent_crime,549813.0,12.0190,26.7323,0.00,0.00,4.00,15.0000,1.0980e+03
propertycrime,549813.0,22.7628,21.5091,0.00,6.00,21.00,32.0000,3.8900e+02
grossannualpay,546938.0,31665.3124,4436.9589,21606.00,28638.00,30726.00,33821.0000,5.4710e+04
jobdensity,500816.0,1.0860,4.1976,0.38,0.54,0.62,0.9300,8.4360e+01
hoursworked,549813.0,37.4853,0.7041,35.00,37.50,37.50,37.5000,3.9900e+01
unemployment,548230.0,8.2427,2.2886,3.40,6.45,8.05,9.8625,1.5100e+01


,missing_share
jobdensity,8.9116e-02
grossannualpay,5.2291e-03
unemployment,2.8792e-03
percent_noedu,1.2732e-05
lone_hh,1.2732e-05
percent_minorities,1.2732e-05
duration,0.0000e+00
age,0.0000e+00
OA01CD,0.0000e+00
OA11CD,0.0000e+00


Showing 20 of 51 rows; the complete table is saved as missingness.csv.


,sales,median_price,lsoas
year,,,
2010,91085,286000.0,4801
2011,89499,291000.0,4816
2012,91980,300000.0,4817
2013,110733,323000.0,4825
2014,117519,365000.0,4831
2015,48997,388000.0,4768


,sales
propertyty,
D,26815
F,290821
S,80643
T,151534


allcrime           violent_crime           propertycrime         
               mean       std          mean       std          mean      std
lsoacode                                                                    
E01000001  157.3953  103.5320        5.4186    5.8599       10.1105   8.3613
E01000002  249.7535  130.9529        8.4113    6.6320       12.1312   6.6416
E01000003   61.9022   39.5310        4.4402    5.8437        4.5598   2.5876
E01000005  426.1622  180.3427       28.8108   21.1960       30.2432  14.9562
E01000006   86.4937   52.7825        5.7722    6.5417       12.5316   7.8933
E01000007  329.1915  155.3173       17.0000   28.4307       28.3830  13.3670
E01000008  231.6875  153.0847       12.4062   14.3838       16.4062  11.4634
E01000009  284.9750  165.8692       24.9000   31.2813       24.6375  18.7935
E01000010  899.0639  681.0678       88.6128  111.2071       46.5376  39.0603
E01000011  148.5484  118.7892       10.0161   14.6024       14.2419  11.8220
E01000012  164.5077  126.4329        9.3692   15.5418       18.4308  14.2544
E01000013  217.9610  124.5075       24.0130   26.8031       27.4545  15.8634
E01000014  164.5778   88.2986       12.2667   13.8026       24.8778  15.3841
E01000015  124.5521   98.6210       12.8438   19.2989       20.8958  17.3342
E01000016  129.6974   70.8856       12.1053   14.8239       21.3026  11.2214
E01000017   75.5082   40.2797        6.3279    6.7025       14.6230   9.8254
E01000018  111.4923   66.1393       12.5077   15.3033       17.8615  10.6915
E01000019  148.7297   92.8520        8.0946    9.8720       24.0270  13.6070
E01000020   91.6364   54.9270        8.7792   10.2104       16.5455   9.7583
E01000021  143.8269   85.5789       12.1154   17.0727       28.6346  15.1942

Showing 20 of 4836 rows; the complete table is saved as within_lsoa_crime_variation.csv.


/tmp/ipykernel_3813217/2320460953.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUT / 'original_data_audit.png', dpi=160); plt.show()


## 2. Monthly source: dates, categories and geographical coverage

Monthly headers are sorted chronologically. The first available month is April 2010. A complete previous-twelve-month measure first exists for April 2011 sales; complete annual 2010 totals remain missing. Recorded zeros are valid observations.


In [6]:
if not MONTHLY_FILE.is_file():
    raise FileNotFoundError('Stage 3 needs MPS LSOA Level Crime (Historical).csv in the project folder or project_data/.')
MPS = MONTHLY_FILE
raw = pd.read_csv(MPS, encoding='utf-8-sig')
raw.columns = raw.columns.str.strip()
if raw.empty or not {'LSOA Code', 'Major Category', 'Minor Category', 'Borough'} <= set(raw.columns):
    raise ValueError(f'Invalid or empty monthly MPS CSV: {MPS}. Restore the original CSV before rebuilding the merged data.')
print('Monthly file:', MPS)
print('SHA256:', hashlib.sha256(MPS.read_bytes()).hexdigest())
print('Category rows:', len(raw))


Monthly file: /home/rachmiel/he3027/project_data/MPS LSOA Level Crime (Historical).csv
SHA256: 9849c110ebb69c0c518126c09a60aa6f7fa5f5294ce22b28628c2e04c24de4fa
Category rows: 112977


In [7]:
required = ['LSOA Code', 'Borough', 'Major Category', 'Minor Category']
assert set(required) <= set(raw.columns)
assert not raw.duplicated(['LSOA Code', 'Major Category', 'Minor Category']).any()
assert not raw[required].isna().any().any()
assert raw.groupby('LSOA Code').Borough.nunique().max() == 1
month_cols = sorted((c for c in raw if re.fullmatch('20\\d{2}(0[1-9]|1[0-2])', c)))
study_months = [c for c in month_cols if '201004' <= c <= '201512']
expected = pd.period_range('2010-04', '2015-12', freq='M').strftime('%Y%m').tolist()
assert study_months == expected, 'Missing study months: ' + str(set(expected) - set(study_months))
counts = raw[study_months].apply(pd.to_numeric, errors='raise')
assert counts.notna().all().all(), 'Inspect missing monthly cells before aggregation'
assert counts.ge(0).all().all() and (counts % 1).eq(0).all().all()
print('Source coverage:', month_cols[0], month_cols[-1], '; study months:', len(study_months))
table(raw.groupby('Major Category').agg(rows=('LSOA Code', 'size'), lsoas=('LSOA Code', 'nunique')), 'source_categories')
table(raw[['Major Category', 'Minor Category']].drop_duplicates().sort_values(['Major Category', 'Minor Category']), 'source_minor_categories')


Source coverage: 201004 202211 ; study months: 69


,rows,lsoas
Major Category,,
ARSON AND CRIMINAL DAMAGE,9543,4988
BURGLARY,9974,4988
DRUG OFFENCES,9834,4988
MISCELLANEOUS CRIMES AGAINST SOCIETY,4983,4983
POSSESSION OF WEAPONS,4875,4875
PUBLIC ORDER OFFENCES,15504,4988
ROBBERY,9028,4983
THEFT,19026,4988
VEHICLE OFFENCES,19045,4988


,Major Category,Minor Category
0,ARSON AND CRIMINAL DAMAGE,ARSON
1,ARSON AND CRIMINAL DAMAGE,CRIMINAL DAMAGE
2,BURGLARY,BURGLARY BUSINESS AND COMMUNITY
3,BURGLARY,BURGLARY IN A DWELLING
4,DRUG OFFENCES,POSSESSION OF DRUGS
5,DRUG OFFENCES,TRAFFICKING OF DRUGS
6,MISCELLANEOUS CRIMES AGAINST SOCIETY,MISC CRIMES AGAINST SOCIETY
7,POSSESSION OF WEAPONS,POSSESSION OF WEAPONS
8,PUBLIC ORDER OFFENCES,OTHER OFFENCES PUBLIC ORDER
9,PUBLIC ORDER OFFENCES,PUBLIC FEAR ALARM OR DISTRESS


Showing 20 of 25 rows; the complete table is saved as source_minor_categories.csv.


In [8]:
GROUPS = {'mps_all': None, 'mps_violence': ['VIOLENCE AGAINST THE PERSON'], 'mps_property': ['BURGLARY', 'THEFT', 'VEHICLE OFFENCES', 'ARSON AND CRIMINAL DAMAGE']}
series = {}
for name, categories in GROUPS.items():
    subset = raw if categories is None else raw.loc[raw['Major Category'].isin(categories)]
    wide = subset.groupby('LSOA Code')[study_months].sum(min_count=1)
    if categories is not None:
        complete = subset.groupby('LSOA Code')['Major Category'].nunique().eq(len(categories))
        wide = wide.where(complete, axis=0)
    series[name] = wide
IV_CATEGORIES = {
    'mps_residential_burglary': 'BURGLARY IN A DWELLING',
    'mps_business_burglary': 'BURGLARY BUSINESS AND COMMUNITY',
}
for name, category in IV_CATEGORIES.items():
    subset = raw.loc[raw['Minor Category'].eq(category)]
    if subset.empty:
        raise ValueError(f'Missing IV offence category in monthly source: {category}')
    series[name] = subset.groupby('LSOA Code')[study_months].sum(min_count=1)
monthly_parts = []
for name, wide in series.items():
    long = wide.reset_index().melt(id_vars='LSOA Code', var_name='month_text', value_name=name)
    long['month'] = pd.to_datetime(long.pop('month_text'), format='%Y%m').dt.to_period('M')
    long = long.rename(columns={'LSOA Code': 'lsoacode'}).set_index(['lsoacode', 'month'])
    monthly_parts.append(long)
monthly = pd.concat(monthly_parts, axis=1).sort_index()
MEASURES = list(series)
table(monthly.groupby(level='month')[MEASURES].sum(min_count=1), 'monthly_source_totals')
monthly.reset_index().to_csv(OUT / 'lsoa_month_crime.csv.gz', index=False, compression={'method': 'gzip', 'compresslevel': 1})


,mps_all,mps_violence,mps_property,mps_residential_burglary,mps_business_burglary
month,,,,,
2010-04,64053,11648,42093,4307,2792.0
2010-05,65951,12657,42334,4385,2759.0
2010-06,66686,12830,43317,4380,2761.0
2010-07,67704,12934,43497,4442,2643.0
2010-08,62821,11251,40534,4517,2685.0
2010-09,63553,11228,41953,4725,2667.0
2010-10,66031,11332,43750,5057,2813.0
2010-11,64530,10578,43700,6008,2851.0
2010-12,54913,9959,36646,5523,2383.0


Showing 20 of 69 rows; the complete table is saved as monthly_source_totals.csv.


In [9]:
grid = pd.MultiIndex.from_product([monthly.index.get_level_values('lsoacode').unique(), pd.period_range('2010-04', '2015-12', freq='M')], names=['lsoacode', 'month'])
monthly = monthly.reindex(grid)
exposure = monthly.copy()
for c in MEASURES:
    exposure[c + '_prior12'] = monthly.groupby(level='lsoacode')[c].transform(lambda s: s.shift(1).rolling(12, min_periods=12).sum())
exposure = exposure.reset_index().rename(columns={'month': 'sale_month'})
borough = raw[['LSOA Code', 'Borough']].drop_duplicates().rename(columns={'LSOA Code': 'lsoacode', 'Borough': 'mps_borough'})
df['sale_month'] = df.transactiondates.dt.to_period('M')
merged = df.merge(borough, on='lsoacode', how='left', validate='many_to_one')
merged = merged.merge(exposure, on=['lsoacode', 'sale_month'], how='left', validate='many_to_one', indicator='monthly_merge')
merged['mps_code_match'] = merged.lsoacode.isin(borough.lsoacode)
merged['mps_sale_month_match'] = merged.monthly_merge.eq('both')
merged['mps_borough_match'] = merged.oslaua.eq(merged.mps_borough)
merged['city_of_london'] = merged.oslaua.eq('E09000001')
merged['mps_geography_ok'] = merged.mps_code_match & merged.mps_borough_match & ~merged.city_of_london


In [10]:
a = monthly.reset_index()
a['year'] = a.month.dt.year
group = a.groupby(['lsoacode', 'year'])
annual = group[MEASURES].sum(min_count=12).where(group[MEASURES].count().eq(12))
annual = annual.rename(columns={c: c + '_annual' for c in MEASURES}).reset_index()
merged = merged.merge(annual, on=['lsoacode', 'year'], how='left', validate='many_to_one')
external_columns = MEASURES + [c + '_prior12' for c in MEASURES] + [c + '_annual' for c in MEASURES]
merged.loc[~merged.mps_geography_ok, external_columns] = np.nan
assert len(merged) == len(df) and merged.transaction_id.is_unique
assert merged.loc[merged.sale_month.lt(pd.Period('2011-04', 'M')), 'mps_all_prior12'].isna().all()
assert merged.loc[merged.year.eq(2010), 'mps_all_annual'].isna().all()
assert merged.loc[merged.city_of_london, external_columns].isna().all().all()
coverage = merged.groupby('year').agg(N=('price', 'size'), code_match=('mps_code_match', 'mean'), geography_ok=('mps_geography_ok', 'mean'), prior12_available=('mps_all_prior12', lambda x: x.notna().mean()), annual_available=('mps_all_annual', lambda x: x.notna().mean()))
table(coverage, 'merge_coverage_by_year')
table(merged.groupby('la_name').agg(N=('price', 'size'), code_match=('mps_code_match', 'mean'), geography_ok=('mps_geography_ok', 'mean')), 'coverage_by_authority')
table(merged.loc[merged.mps_code_match & ~merged.mps_borough_match].groupby(['lsoacode', 'la_name', 'mps_borough']).size().to_frame('N'), 'borough_mismatch_transactions')
table(merged.loc[~merged.mps_code_match].groupby(['lsoacode', 'la_name']).size().to_frame('N'), 'unmatched_lsoas')
merged.to_csv(OUT / 'land_registry_crime_monthly_merged.csv.gz', index=False, compression={'method': 'gzip', 'compresslevel': 1})
print('Saved all', len(merged), 'transactions with monthly, annual and prior12 measures.')
print('Usable prior12 exposures:', merged.mps_all_prior12.notna().sum())


,N,code_match,geography_ok,prior12_available,annual_available
year,,,,,
2010,91085,0.9404,0.9403,0.0000,0.0000
2011,89499,0.9378,0.9377,0.7460,0.9377
2012,91980,0.9286,0.9286,0.9286,0.9286
2013,110733,0.9240,0.9240,0.9240,0.9240
2014,117519,0.9211,0.9211,0.9211,0.9211
2015,48997,0.9256,0.9256,0.9256,0.9256


,N,code_match,geography_ok
la_name,,,
Barking and Dagenham,9080,0.8529,0.8529
Barnet,23699,0.9129,0.9129
Bexley,16683,0.9717,0.9717
Brent,12985,0.9630,0.9630
Bromley,27399,0.9870,0.9870
Camden,14419,0.9570,0.9570
City of London,1583,0.0000,0.0000
Croydon,23731,0.9267,0.9267
Ealing,18036,0.9581,0.9580


Showing 20 of 38 rows; the complete table is saved as coverage_by_authority.csv.


,,,N
lsoacode,la_name,mps_borough,
E01000804,Lewisham,E09000006,9
E01001071,Tandridge,E09000008,1
E01002564,Ealing,E09000018,2


,,N
lsoacode,la_name,
E01000001,City of London,172
E01000002,City of London,564
E01000003,City of London,184
E01000005,City of London,37
E01000010,Barking and Dagenham,266
E01000023,Barking and Dagenham,587
E01000048,Barking and Dagenham,9
E01000092,Barking and Dagenham,332
E01000109,Barking and Dagenham,85


Showing 20 of 186 rows; the complete table is saved as unmatched_lsoas.csv.


Saved all 549813 transactions with monthly, annual and prior12 measures.
Usable prior12 exposures: 408094


In [11]:
PAIRS = {'allcrime': 'mps_all', 'violent_crime': 'mps_violence', 'propertycrime': 'mps_property'}
area = merged.loc[merged.mps_geography_ok].groupby(['lsoacode', 'year'], as_index=False).first()
compare = []
for supplied, ext in PAIRS.items():
    for year, b in area.groupby('year'):
        valid = b[[supplied, ext + '_annual']].dropna()
        compare.append({'supplied': supplied, 'external': ext, 'year': year, 'lsoa_years': len(valid), 'supplied_mean': valid[supplied].mean(), 'external_mean': valid[ext + '_annual'].mean(), 'supplied_zero_share': valid[supplied].eq(0).mean(), 'external_zero_share': valid[ext + '_annual'].eq(0).mean(), 'correlation': valid[supplied].corr(valid[ext + '_annual']) if valid[supplied].nunique() > 1 and len(valid) > 2 else np.nan, 'exact_match_share': valid[supplied].eq(valid[ext + '_annual']).mean()})
table(pd.DataFrame(compare), 'supplied_vs_external_annual')


,supplied,external,year,lsoa_years,supplied_mean,external_mean,supplied_zero_share,external_zero_share,correlation,exact_match_share
0,allcrime,mps_all,2010,0,NaN,NaN,NaN,NaN,NaN,NaN
1,allcrime,mps_all,2011,4638,243.0226,149.3862,0.0000,0.0000,0.9482,0.0017
2,allcrime,mps_all,2012,4637,221.8352,143.3882,0.0000,0.0000,0.9590,0.0030
3,allcrime,mps_all,2013,4642,195.3175,133.9978,0.0000,0.0000,0.9646,0.0062
4,allcrime,mps_all,2014,4648,184.8853,132.8201,0.0000,0.0000,0.9645,0.0054
5,allcrime,mps_all,2015,4590,14.2425,139.8115,0.0000,0.0000,0.9322,0.0000
6,violent_crime,mps_violence,2010,0,NaN,NaN,NaN,NaN,NaN,NaN
7,violent_crime,mps_violence,2011,4638,29.5082,25.3894,0.0015,0.0026,0.9503,0.0755
8,violent_crime,mps_violence,2012,4637,26.9827,24.8631,0.0009,0.0011,0.9522,0.0766
9,violent_crime,mps_violence,2013,4642,8.2355,24.0952,0.0282,0.0011,0.9047,0.0114


## 3. Compare annual counts on identical LSOAs

Only complete external January-December totals are compared with supplied annual values. There is no annual-versus-single-month match statistic. A large difference need not be a pure recording error: offence definitions can differ. Annual 2010 comparisons have zero valid pairs.


In [12]:
comparison = pd.DataFrame(compare)
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, supplied in zip(axes, CRIMES):
    b = comparison.loc[comparison.supplied.eq(supplied)].set_index('year')
    b[['supplied_mean','external_mean']].plot(ax=ax, marker='o', title=supplied)
    ax.set_ylabel('Mean offences per matched LSOA')
fig.tight_layout(); fig.savefig(OUT / 'annual_source_comparison.png', dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(12,4))
monthly.groupby(level='month')[MEASURES[:3]].sum(min_count=1).rename_axis('month').plot(ax=ax)
ax.set_title('Published MPS monthly totals across source LSOAs')
ax.set_ylabel('Offences'); fig.tight_layout(); fig.savefig(OUT / 'mps_monthly_trends.png', dpi=160); plt.show()


/tmp/ipykernel_3813217/2327138875.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUT / 'annual_source_comparison.png', dpi=160); plt.show()
/tmp/ipykernel_3813217/2327138875.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_ylabel('Offences'); fig.tight_layout(); fig.savefig(OUT / 'mps_monthly_trends.png', dpi=160); plt.show()


## 4. Construct calendar-year lags and export

Calendar lags use explicit LSOA/year joins, never shifts of transaction rows. The merged file contains original annual counts, original previous-year counts, MPS annual counts, MPS previous-year counts and sale-specific prior12 counts. MPS coverage flags are retained. An original lag is unavailable if that LSOA-year is absent from the original source.

The current-year and previous-year original-data notebooks use the same comparison sample by requiring both original exposures for all categories. The MPS notebook uses one common sample across all three MPS exposures. Different source samples must not be interpreted as a pure measurement experiment.


In [13]:
for c in CRIMES:
    merged[c + '_annual'] = merged[c]
original_lags = area_year.rename(columns={c:c + '_lagyear' for c in CRIMES}).copy()
original_lags['year'] += 1
merged = merged.merge(original_lags, on=['lsoacode','year'], how='left', validate='many_to_one')
mps_names = ['mps_all','mps_violence','mps_property']
lag_measures = mps_names + list(IV_CATEGORIES)
mps_lags = annual[['lsoacode','year'] + [c + '_annual' for c in lag_measures]].copy()
mps_lags['year'] += 1
mps_lags = mps_lags.rename(columns={c + '_annual':c + '_lagyear' for c in lag_measures})
merged = merged.merge(mps_lags, on=['lsoacode','year'], how='left', validate='many_to_one')
merged.loc[~merged.mps_geography_ok, [c + '_lagyear' for c in lag_measures]] = np.nan
london = merged.oslaua.str.startswith('E090', na=False)
borough_counts = merged.loc[london].groupby('lsoacode').oslaua.nunique()
merged['analysis_geography_ok'] = london & ~merged.lsoacode.isin(borough_counts.index[borough_counts.gt(1)])
original_required = [c + '_' + e for c in CRIMES for e in ['annual','lagyear']]
mps_required = [c + '_' + e for c in mps_names for e in ['annual','lagyear','prior12']]
merged['original_pair_available'] = merged[original_required].notna().all(axis=1)
merged['mps_exposures_available'] = merged[mps_required].notna().all(axis=1) & merged.mps_geography_ok
assert len(merged) == len(df) and merged.transaction_id.is_unique
assert merged.loc[merged.year.eq(2010),[c + '_lagyear' for c in CRIMES]].isna().all().all()
assert merged.loc[merged.year.le(2011),[c + '_lagyear' for c in mps_names]].isna().all().all()
table(merged.groupby('year').agg(sales=('price','size'), original_pair_available=('original_pair_available','mean'), mps_exposures_available=('mps_exposures_available','mean')), 'final_exposure_coverage')
# Stable names consumed by notebook 04; counts are stored before log transforms.
iv_columns = []
for alias, measure in [('residential', 'mps_residential_burglary'), ('business', 'mps_business_burglary')]:
    for exposure_name in ['annual', 'lagyear', 'prior12']:
        column = alias + '_' + exposure_name
        merged[column] = merged[measure + '_' + exposure_name]
        iv_columns.append(column)
merged['iv_exposures_available'] = merged[iv_columns].notna().all(axis=1) & merged.mps_geography_ok
assert merged.loc[~merged.mps_geography_ok, iv_columns].isna().all().all()
assert merged.loc[merged.year.le(2011), ['residential_lagyear', 'business_lagyear']].isna().all().all()
assert not merged[iv_columns].lt(0).any().any(), 'Negative observed IV crime count'
table(merged.groupby('year').iv_exposures_available.mean().to_frame(), 'iv_exposure_coverage')
merged_path = DATA_DIR / 'crime_housing_merged.csv.gz'
merged.to_csv(merged_path, index=False, compression={'method':'gzip','compresslevel':1})
monthly.reset_index().to_csv(DATA_DIR / 'mps_monthly_panel.csv.gz', index=False, compression={'method':'gzip','compresslevel':1})
manifest = {'rows':len(merged), 'original_sha256':hashlib.sha256(DATA.read_bytes()).hexdigest(), 'monthly_sha256':hashlib.sha256(MONTHLY_FILE.read_bytes()).hexdigest(), 'schema_version':2, 'iv_columns':iv_columns}
import json
(DATA_DIR / 'crime_housing_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved:', merged_path, '\nRun notebooks 01–04 next; IV exposures are included in this merged file.')


,sales,original_pair_available,mps_exposures_available
year,,,
2010,91085,0.0000,0.0000
2011,89499,0.9958,0.0000
2012,91980,0.9992,0.9286
2013,110733,0.9991,0.9240
2014,117519,0.9998,0.9211
2015,48997,1.0000,0.9256


,iv_exposures_available
year,
2010,0.0000
2011,0.0000
2012,0.9284
2013,0.9239
2014,0.9209
2015,0.9254


Saved: /home/rachmiel/he3027/data/crime_housing_merged.csv.gz 
Run notebooks 01–04 next; IV exposures are included in this merged file.
